In [33]:
from pyspark.sql.session import SparkSession


spark = (

 SparkSession
 .builder
 .appName("way-to-create-dataframe")
 .master("local")
 .getOrCreate()   
)

In [34]:
fruits_list = [["apple","banana","orange","mango"]]

spark.createDataFrame(fruits_list).show()

+-----+------+------+-----+
|   _1|    _2|    _3|   _4|
+-----+------+------+-----+
|apple|banana|orange|mango|
+-----+------+------+-----+



In [35]:
spark.createDataFrame(fruits_list,["name", "name1", "name2", "name3"]).show()

+-----+------+------+-----+
| name| name1| name2|name3|
+-----+------+------+-----+
|apple|banana|orange|mango|
+-----+------+------+-----+



In [36]:
fruits_list = [["apple", "banana", "orange", "mango"]]

df = spark.createDataFrame(
    fruits_list, 
    "name STRING, name1 STRING, name2 STRING, name3 STRING"
)
df.show()


+-----+------+------+-----+
| name| name1| name2|name3|
+-----+------+------+-----+
|apple|banana|orange|mango|
+-----+------+------+-----+



In [38]:
### creating one column

fruits_list = [ (1, 'apple'), (2, 'banana'), (3, 'orange')]

spark.createDataFrame(fruits_list, "id int, fruit string").show()

+---+------+
| id| fruit|
+---+------+
|  1| apple|
|  2|banana|
|  3|orange|
+---+------+



['apple', 'banana', 'orange']


### toDF() vs. createDataFrame()

- `rdd.toDF():` Simpler syntax; works directly on an RDD; always infers data types automatically; cannot enforce a strict StructType schema on raw tuples.
- `spark.createDataFrame():` Works on local Python lists, Pandas DataFrames, and RDDs; allows you to pass an explicit StructType or DDL schema to enforce exact data types.


In [50]:
### using context

fruits_list = [['apple','banana','orange']]

parallel_collection = spark.sparkContext.parallelize(fruits_list)

parallel_collection.collect()

parallel_collection.toDF(["name","name1","name2"]).show()

+-----+------+------+
| name| name1| name2|
+-----+------+------+
|apple|banana|orange|
+-----+------+------+



In [51]:

fruits_list = [['apple'],['banana'],['orange']]

parallel_collection = spark.sparkContext.parallelize(fruits_list)

parallel_collection.collect()

parallel_collection.toDF(["fruit"]).show()

+------+
| fruit|
+------+
| apple|
|banana|
|orange|
+------+



In [53]:
parallel_collection.toDF().show()

+------+
|    _1|
+------+
| apple|
|banana|
|orange|
+------+



In [52]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ToDFExample").getOrCreate()

# Create an RDD of tuples
rdd = spark.sparkContext.parallelize([
    ("Alice", 25, "Engineering"),
    ("Bob", 30, "Marketing")
])

# Convert RDD to DataFrame with column names
df = rdd.toDF(["name", "age", "department"])

df.printSchema()
df.show()


26/11/12 12:39:41 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)

+-----+---+-----------+
| name|age| department|
+-----+---+-----------+
|Alice| 25|Engineering|
|  Bob| 30|  Marketing|
+-----+---+-----------+



In [54]:
# Rename all 3 columns at once
df_renamed = df.toDF("employee_name", "employee_age", "dept")
df_renamed.show()


+-------------+------------+-----------+
|employee_name|employee_age|       dept|
+-------------+------------+-----------+
|        Alice|          25|Engineering|
|          Bob|          30|  Marketing|
+-------------+------------+-----------+



## createDataFrame Example:

### 1. createDataFrame(rows: List[Row], schema: StructType)

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

rows = [
    Row(name="Alice", age=25),
    Row(name="Bob", age=30)
]

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = spark.createDataFrame(rows, schema=schema)
df.show()


+-----+---+
| name|age|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+



### 2. createDataFrame(rowRDD: RDD[Row], schema: StructType) and JavaRDD[Row]


In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

rdd = spark.sparkContext.parallelize([
    Row(name="Alice", age=25),
    Row(name="Bob", age=30)
])

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = spark.createDataFrame(rdd, schema=schema)
df.show()


### 3. createDataFrame(rdd: RDD[A])(TypeTag[A])
- From a Typed RDD (Schema Inferred Automatically)

In [ ]:
from pyspark.sql import Row

# Using Row objects (column names and types are inferred)
rdd = spark.sparkContext.parallelize([
    Row(name="Alice", age=25),
    Row(name="Bob", age=30)
])

df = spark.createDataFrame(rdd)
df.show()


+-----+---+
| name|age|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+





### 4. createDataFrame(data: Seq[A])(TypeTag[A])

From a Python List / Sequence (Schema Inferred Automatically)

In [61]:

# Option A: List of Row objects (infers column names and types)
from pyspark.sql import Row
data_rows = [Row(name="Alice", age=25), Row(name="Bob", age=30)]
df1 = spark.createDataFrame(data_rows)

df1.show()

# Option B: List of tuples (infers types, assigns _1, _2 unless column names are given)
data_tuples = [("Alice", 25), ("Bob", 30)]
df2 = spark.createDataFrame(data_tuples, ["name", "age"])
df2.show()


df3 = spark.createDataFrame(data_tuples)
df3.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+

+-----+---+
| name|age|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+

+-----+---+
|   _1| _2|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+




### 5. JavaBean Equivalents (beanClass: Class[_])
 createDataFrame(rdd: RDD[_], beanClass)

 and createDataFrame(data: List[_], beanClass)
Python does not have JavaBeans. 
Instead, PySpark achieves the same structured object behavior using Python dataclasses, namedtuple, or dictionaries:

In [44]:
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

people = [Person("Alice", 25), Person("Bob", 30)]

# Works with both a Python list or an RDD of dataclass objects
df_from_list = spark.createDataFrame(people)
df_from_rdd = spark.createDataFrame(spark.sparkContext.parallelize(people))

df_from_list.show()


+---+-----+
|age| name|
+---+-----+
| 25|Alice|
| 30|  Bob|
+---+-----+



In [55]:
spark.sparkContext.parallelize(people).collect()

[Person(name='Alice', age=25), Person(name='Bob', age=30)]

In [58]:
spark.sparkContext.parallelize(people).toDF().show() ## toDF works on parallelize collection in pyspark

+---+-----+
|age| name|
+---+-----+
| 25|Alice|
| 30|  Bob|
+---+-----+



In [ ]:
spark.stop()

: 